** By doing lot of data engineering we can create lot of features but not all features created are useful and not optimal way
- Simplest way to select features would be to remove features with very low variance
- very low variance (i.e. very close to 0), they
are close to being constant and thus, do not add any value to any model at all
- Scikit-learn has an
implementation for VarianceThreshold 

In [5]:
from sklearn.feature_selection import VarianceThreshold
var_thresh = VarianceThreshold(threshold=0.1)
transformed_data = var_thresh.fit_transform(df)
# transformed data will have all columns with variance less
# than 0.1 removed

ValueError: No feature in X meets the variance threshold 0.10000

In [8]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# Small dataset
data = {
    "A": [1, 1, 1, 1, 1],   # Zero variance (constant column)
    "B": [0, 1, 0, 1, 0],   # Some variance
    "C": [10, 11, 10, 11, 10],  # Small variance
    "D": [5, 20, 15, 30, 25]    # Larger variance
}

df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)

# Apply VarianceThreshold
var_thresh = VarianceThreshold(threshold=0.1)  # remove features with variance < 0.1
transformed_data = var_thresh.fit_transform(df)

# Get the selected columns
selected_columns = df.columns[var_thresh.get_support()] #??? get_suppport
df_transformed = pd.DataFrame(transformed_data, columns=selected_columns)

print("\nSelected DataFrame after VarianceThreshold:")
print(df_transformed)


Original DataFrame:
   A  B   C   D
0  1  0  10   5
1  1  1  11  20
2  1  0  10  15
3  1  1  11  30
4  1  0  10  25

Selected DataFrame after VarianceThreshold:
   B   C   D
0  0  10   5
1  1  11  20
2  0  10  15
3  1  11  30
4  0  10  25


In [7]:
df_transformed

,B,C,D
0,0,10,5
1,1,11,20
2,0,10,15
3,1,11,30
4,0,10,25


** - We can also remove high correlation 
- we can use the :-
#### Pearson Correlation

In [15]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# fetch a regression dataset
data=fetch_california_housing()
X=data['data']
col_names=data['feature_names']
y=data['target']
df = pd.DataFrame(X, columns=col_names)
# introduce a highly correlated column
df.loc[:, "MedInc_Sqrt"] = df.MedInc.apply(np.sqrt)
# get correlation matrix (pearson)
df.corr()
 # MedInc_Sqrt has a very high correlation with MedInc

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_Sqrt
MedInc,1.000000,-0.119034,0.326895,-0.062040,0.004834,0.018766,-0.079809,-0.015176,0.984329
HouseAge,-0.119034,1.000000,-0.153277,-0.077747,-0.296244,0.013191,0.011173,-0.108197,-0.132797
AveRooms,0.326895,-0.153277,1.000000,0.847621,-0.072213,-0.004852,0.106389,-0.027540,0.326688
AveBedrms,-0.062040,-0.077747,0.847621,1.000000,-0.066197,-0.006181,0.069721,0.013344,-0.066910
Population,0.004834,-0.296244,-0.072213,-0.066197,1.000000,0.069863,-0.108785,0.099773,0.018415
AveOccup,0.018766,0.013191,-0.004852,-0.006181,0.069863,1.000000,0.002366,0.002476,0.015266
Latitude,-0.079809,0.011173,0.106389,0.069721,-0.108785,0.002366,1.000000,-0.924664,-0.084303
Longitude,-0.015176,-0.108197,-0.027540,0.013344,0.099773,0.002476,-0.924664,1.000000,-0.015569
MedInc_Sqrt,0.984329,-0.132797,0.326688,-0.066910,0.018415,0.015266,-0.084303,-0.015569,1.000000


## Unvariate Feature Selection

- some univariate ways of feature selection. Univariate
feature selection is nothing but a scoring of each feature against a given target
- Mutual information, ANOVA F-test and chi2 are some of the most popular
methods for univariate feature selection.
- There are two ways of using these in scikit-learn.
- - SelectKBest: It keeps the top-k scoring features
- - SelectPercentile: It keeps the top features which are in a percentage
specified by the user

** It must be noted that you can use chi2 only for data which is non-negative in nature.
This is a particularly useful feature selection technique in natural language
processing when we have a bag of words or tf-idf based features

In [17]:
from sklearn.feature_selection import chi2
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SelectPercentile



class UnivariateFeatureSelection:
    def __init__(self, n_features, problem_type, scoring):
        """
        Custom univariate feature selection wrapper on
        different univariate feature selection models from
        scikit-learn.
        :param n_features: SelectPercentile if float else SelectKBest
        :param problem_type: classification or regression
        :param scoring: scoring function, string
        """
        # for a given problem type, there are only
        # a few valid scoring methods
        # you can extend this with your own custom
        # methods if you wish
        if problem_type == "classification":
            valid_scoring = {
                            "f_classif": f_classif,
                            "chi2": chi2,
                            "mutual_info_classif": mutual_info_classif
                            }
        else:
            valid_scoring = {
                            "f_regression": f_regression,
                            "mutual_info_regression": mutual_info_regression
                            }
        # raise exception if we do not have a valid scoring method
        if scoring not in valid_scoring:
            raise Exception("Invalid scoring function")
        # if n_features is int, we use selectkbest
        # if n_features is float, we use selectpercentile
        # please note that it is int in both cases in sklearn
        if isinstance(n_features, int):
            self.selection = SelectKBest(
                             valid_scoring[scoring],
                             k=n_features
            )
        elif isinstance(n_features, float):
            self.selection = SelectPercentile(
                            valid_scoring[scoring],
                            percentile=int(n_features * 100)
            )
        else:
            raise Exception("Invalid type of feature")
            # same fit function
    def fit(self, X, y):
        return self.selection.fit(X, y)
    # same transform function
    def transform(self, X):
        return self.selection.transform(X)
    # same fit_transform function
    def fit_transform(self, X, y):
        return self.selection.fit_transform(X, y)



In [19]:
ufs = UnivariateFeatureSelection(
n_features=0.1,
problem_type="regression",
scoring="f_regression"
)
ufs.fit(X, y)
X_transformed = ufs.transform(X)

In [20]:
X_transformed

array([[8.3252],
       [8.3014],
       [7.2574],
       ...,
       [1.7   ],
       [1.8672],
       [2.3886]])

** 3) Fit / Transform / Fit-Transform
- ufs.fit(X, y) → computes per-feature scores between each X column and y (using chosen scoring func).
- ufs.transform(X) → returns a NumPy array with only the selected features (like sklearn selectors).
- ufs.fit_transform(X, y) → shorthand for fit then transform.
- Note: transform() returns a NumPy array. If you want a DataFrame with column names, see step 6.

    
** 4) Important practical details & gotchas

- chi2 requires non-negative features (counts or frequencies). If you use chi2, scale or transform inputs (e.g. MinMaxScaler) or avoid negative values.
- f_classif / f_regression are ANOVA/F-test based — work on numeric continuous features.
- mutual_info_* are nonparametric, capture non-linear relationships, often slower but useful.
- If n_features is a float, it must be between 0.0 and 1.0 (converted to percentile *100).
- The selector computes univariate scores — it does not account for feature interactions; good as a fast filter before more expensive models.
- Complexity: very fast — roughly O(n_features × n_samples) for scoring each feature.

** 5) Extra useful attributes after fit

- After ufs.fit(X, y) you can inspect:
- ufs.selection.scores_ — raw scores for each feature (if the scoring function provides)
- ufs.selection.pvalues_ — (for F tests) p-values (if available)
- ufs.selection.get_support() — boolean mask of selected features

** Quick tips

- If you plan to pipeline this into sklearn.pipeline.Pipeline, the fact that transform returns a NumPy array is fine — downstream transformers usually accept arrays. If you want to keep DataFrame columns end-to-end, wrap a small transformer to convert back to DataFrame.
- For reproducibility and clarity, prefer explicit int for top-k (e.g., k=10) or float for percent (e.g., 0.2 → top 20%).
- If you want n_features to be flexible (like "auto"), add a small helper to compute k based on X.shape[1].

** Live toy run with your UnivariateFeatureSelection class so you can see:
- Input data
- Scores computed
- Selected features

In [23]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes

# load diabetes regression dataset
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Input shape:", X.shape)
print(X.head())


Input shape: (442, 10)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


In [24]:
ufs = UnivariateFeatureSelection(
    n_features=0.3,                 # keep top 30%
    problem_type="regression", 
    scoring="f_regression"
)

X_trans = ufs.fit_transform(X, y)

print("\nTransformed shape:", X_trans.shape)



Transformed shape: (442, 3)


In [25]:
# mask of which features got selected
mask = ufs.selection.get_support()
selected_cols = X.columns[mask]

# scores for each feature
scores = ufs.selection.scores_

# combine into DataFrame
scores_df = pd.DataFrame({
    "feature": X.columns,
    "score": scores,
    "selected": mask
}).sort_values("score", ascending=False)

print("\nScores with selection flag:")
print(scores_df)

# show only selected features
print("\nSelected features:", list(selected_cols))



Scores with selection flag:
  feature       score  selected
2     bmi  230.653764      True
8      s5  207.271194      True
3      bp  106.520131      True
7      s4  100.069264     False
6      s3   81.239659     False
9      s6   75.399683     False
4      s1   20.710567     False
0     age   16.101374     False
5      s2   13.746079     False
1     sex    0.817423     False

Selected features: ['bmi', 'bp', 's5']


### Greedy feature selection

- the first step is to choose a
model. 
- The second step is to select a loss/scoring function. 
- And the third and final
step is to iteratively evaluate each feature and add it to the list of “good” features if
it improves loss/score.
- This feature selection process
will fit a given model each time it evaluates a feature. The computational cost
associated with this kind of method is very high. It will also take a lot of time for
this kind of feature selection to finish. And if you do not use this feature selection
properly, then you might even end up overfitting the model.